# 填充和步幅
---
## 环境配置

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), ".."))
os.environ["TILE_FWK_DEVICE_ID"] = "0"
import pypto
import torch
import torch_npu
import numpy as np

device_id = int(os.environ["TILE_FWK_DEVICE_ID"])
torch.npu.set_device(device_id)
device = f"npu:{device_id}"
mode = pypto.RunMode.NPU

---

## 练习 6.3.1  

对于本节中的最后一个示例，计算其输出形状，以查看它是否与实验结果一致。

### 解答

$$out_{shape}=\lfloor(n_h-k_h+p_h*2+s_h)/s_h\rfloor \times \lfloor(n_w-k_w+p_w*2+s_w)/s_w\rfloor.$$

&emsp;&emsp;示例中 `X.shape = [8, 8]` ，计算得出 $out_shape = [(8-3+0+3)/3, (8-5+2+4)/4] = [2.67, 2.25]$ ，向下取整，所以为 $[2, 2]$。


以下使用 `torch` 编程进行验证：

In [3]:
import torch
from torch import nn

def comp_conv2d(conv2d, X):
    X = X.reshape((1, 1) + X.shape)
    Y = conv2d(X)
    return Y.reshape(Y.shape[2:])

X = torch.rand(size=(8, 8))

conv2d = nn.Conv2d(1, 1, kernel_size=(3, 5), padding=(0, 1), stride=(3, 4))

comp_conv2d(conv2d, X).shape

torch.Size([2, 2])

使用 `PyPTO` 编程进行验证:

In [4]:
from src.PyPTOConvPrimitive import conv2d

X = torch.rand((8, 8), dtype=torch.float32, device=device)
out = conv2d(X, kernel_size=(3, 5), stride=(3, 4), padding=(0, 1))
print(f'PyPTO conv output shape: {out.shape}')

PyPTO conv output shape: torch.Size([2, 2])


---

## 练习 6.3.2

在本节中的实验中，试一试其他填充和步幅组合。

### 解答

&emsp;&emsp; 下面将举两个其他的填充和步幅组合。

&emsp;&emsp;在`padding`大小为 $[1,2]$ ，`stride`大小为 $[2,3]$ 时，输出形状为 $[4,3]$。

&emsp;&emsp;示例中 `X.shape = [8, 8]`，计算得出 $out_{shape} = [(8-3+2+2)/2, (8-5+4+3)/3] = [4.5, 3,33]$ ，向下取整，所以为 $[4, 3]$。

以下使用 `torch` 编程进行验证：

In [5]:
import torch
from torch import nn

def comp_conv2d(conv2d, X):
    X = X.reshape((1, 1) + X.shape)
    Y = conv2d(X)
    return Y.reshape(Y.shape[2:])

X = torch.rand(size=(8, 8))

&emsp;&emsp;在`padding`大小为 $[2,3]$ ，`stride`大小为 $[1,2]$ 时，输出形状为 $[10,5]$ 。

&emsp;&emsp;示例中 `X.shape = [8, 8]` ，计算得出 $out_{shape} = [(8-3+4+1)/1, (8-5+6+2)/2] = [10, 5.5]$ ，向下取整，所以为 $[10, 5]$ 。

In [6]:
conv2d = nn.Conv2d(1, 1, kernel_size=(3, 5), padding=(1, 2), stride=(2, 3))

comp_conv2d(conv2d, X).shape

torch.Size([4, 3])

In [7]:
conv2d = nn.Conv2d(1, 1, kernel_size=(3, 5), padding=(2, 3), stride=(1, 2))

comp_conv2d(conv2d, X).shape

torch.Size([10, 5])

使用 `PyPTO` 编程进行验证:

In [8]:
from src.PyPTOConvPrimitive import conv2d

X = torch.rand((8, 8), dtype=torch.float32, device=device)

out1 = conv2d(X, kernel_size=(3, 5), stride=(2, 3), padding=(1, 2))
print(f'padding=(1,2), stride=(2,3) -> {out1.shape}')

out2 = conv2d(X, kernel_size=(3, 5), stride=(1, 2), padding=(2, 3))
print(f'padding=(2,3), stride=(1,2) -> {out2.shape}')

padding=(1,2), stride=(2,3) -> torch.Size([4, 3])
padding=(2,3), stride=(1,2) -> torch.Size([10, 5])


---

## 练习 6.3.3

对于音频信号，步幅$2$说明什么？

### 解答

&emsp;&emsp;对于音频信号而言，步幅为 $2$ 就是以 $2$ 为周期对信号进行采样计算。

---

## 练习 6.3.4

步幅大于$1$的计算优势是什么？

### 解答

&emsp;&emsp;1.**减少计算量**：步幅大于1意味着卷积核在输入特征图（`feature map`）上的滑动距离更大，因此每次卷积操作涵盖的输入数据更少。这减少了进行卷积操作所需的计算量。\
&emsp;&emsp;2.**降低空间维度**：使用大步幅会减小输出特征图的空间尺寸。例如，在进行下采样（`subsampling`）或降维时，较大的步幅可以有效减小输出数据的高度和宽度，这有助于减少后续层的计算负担。\
&emsp;&emsp;3.**控制过拟合**：通过减少网络中的参数数量和计算量，大步幅有助于控制过拟合。这是因为减少了学习参数，从而降低了模型复杂度。\
&emsp;&emsp;4.**增加感受野**：较大的步幅可以增加网络层的感受野（即网络层可以观察到的输入数据的区域大小）。这意味着网络可以在较低的空间分辨率下捕捉到更广泛的上下文信息。\
&emsp;&emsp;5.**加快网络训练速度**：由于计算量的减少和参数数量的降低，使用较大步幅可以加快网络的训练速度，使得模型更快地收敛。

---
## 参考答案来源
参考答案和 PyTorch 代码实现来源：[https://datawhalechina.github.io/d2l-ai-solutions-manual/#](https://datawhalechina.github.io/d2l-ai-solutions-manual/#)